# Lite-Mono 재현 (Colab)

| 언제 | 셀 |
|---|---|
| 처음 한 번 | A → B → C → D (D는 KITTI 다운로드·변환, 수 시간) |
| 매 세션 | A → B → E, 이후 F/G/H |

- Colab Pro 세션은 보통 최대 12시간이고 브라우저를 닫거나 오래 비우면 끊긴다. G는 매 epoch 끝에 Drive에 저장하므로 **끊기면 A → B → E → G를 다시 실행**하면 마지막 epoch 다음부터 이어간다.
- 모든 결과는 `DRIVE` 아래에 남는다. `/content`는 세션이 끝나면 지워진다.
- 학습은 카메라 영상만 쓴다. LiDAR는 D에서 평가용 정답 깊이를 만들 때만 쓴다.

## A. 설정 (매 세션)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/lite_mono'  # 영구 저장 위치
REPO_URL = 'https://github.com/Sangbum-K/Lite-Mono-Depth-Estimation.git'  # private이면 token 포함 URL 또는 Drive 복사 사용
os.environ.update(DRIVE=DRIVE, REPO_URL=REPO_URL, REPO='/content/Lite-Mono-Depth-Estimation',
                  KITTI='/content/kitti_data')
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%%bash
set -e
[ -d $REPO ] || git clone -q $REPO_URL $REPO
cd $REPO && git log -1 --format='code revision: %H'
pip install -q -r environment/requirements-colab.txt
mkdir -p $DRIVE/env
{ date; nvidia-smi || echo 'no GPU (CPU runtime)'; nproc; free -g; df -h /content; python -V; pip freeze; } > $DRIVE/env/session_$(date +%Y%m%d_%H%M).txt
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'no GPU: CPU runtime is fine for B and D only'
python -c "import torch, torchvision; print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'cuda', torch.version.cuda)"
echo "CPU $(nproc)"; df -h /content | tail -1

## B. 공식 가중치 (처음에만 다운로드, 매 세션 checksum 확인)

In [ ]:
%%bash
set -e
W=$DRIVE/pretrained; mkdir -p $W
[ -f $W/lite-mono-pretrain.pth ] || curl -fsSL -o $W/lite-mono-pretrain.pth https://surfdrive.surf.nl/files/index.php/s/InMMGd5ZP2fXuia/download
[ -f $W/lite-mono_640x192/encoder.pth ] || { curl -fsSL -o /tmp/lm.zip https://surfdrive.surf.nl/files/index.php/s/CUjiK221EFLyXDY/download && unzip -o -q /tmp/lm.zip -d $W; }
cd $W && sha256sum -c <<EOF
9877014dfaede372ca887dea354cb534788eb68b6ded13a9a1d39c715c04fc9f  lite-mono-pretrain.pth
9c5bc031b4a6a8d9c6df5f525c6740594b7d814841cc924f61565d0da624e046  lite-mono_640x192/encoder.pth
c452dd45e23a4cddd823943a43e793537c0095c0c8fc2037a8a4e6ad30acc224  lite-mono_640x192/depth.pth
EOF

## C. Smoke test (데이터 없이, 새 환경에서 한 번)
코드-가중치 일치, ~3.1M 파라미터, 호환 수정, GPU forward/backward를 확인한다.

In [ ]:
!cd $REPO && python scripts/smoke_test.py --pretrained $DRIVE/pretrained

## D. KITTI 준비 (처음 한 번)
**GPU가 필요 없으므로 CPU 런타임에서 실행해 compute unit을 아낀다** (A → B → D). drive 60개 zip(약 175GB)을 하나씩 받아 필요한 프레임만 jpg로 남기고 zip은 지운다. 중간에 끊겨도 다시 실행하면 이어간다.
끝나면 평가용 GT를 만들고, 이미지 묶음과 GT를 Drive에 저장한다. Drive 여유 공간을 먼저 확인할 것.

In [ ]:
%%bash
set -e
cd $REPO
python scripts/prepare_kitti.py --kitti_out $KITTI --velo_out /content/kitti_velo --tmp /content/kitti_zip
# 공식 test GT (raw LiDAR, Eigen 697) -> evaluate_depth.py가 읽는 위치
python vendor/Lite-Mono/export_gt_depth.py --data_path /content/kitti_velo --split eigen
# 이미지별 평가용 sparse GT (test, val)
python analysis/eval_per_image.py build-gt --split_file data/splits/test_files.txt --velo_root /content/kitti_velo --out /content/test_gt.npz
python analysis/eval_per_image.py build-gt --split_file data/splits/val_files.txt --velo_root /content/kitti_velo --out /content/val_gt.npz
mkdir -p $DRIVE/data
cp vendor/Lite-Mono/splits/eigen/gt_depths.npz $DRIVE/data/eigen_gt_depths.npz
cp /content/test_gt.npz /content/val_gt.npz $DRIVE/data/
tar -cf $DRIVE/data/kitti_jpg.tar -C /content kitti_data
du -sh $KITTI /content/kitti_velo $DRIVE/data/*

## E. 데이터 복원 (매 세션)

In [ ]:
%%bash
set -e
[ -d $KITTI/2011_09_26 ] || tar -xf $DRIVE/data/kitti_jpg.tar -C /content
cp $DRIVE/data/eigen_gt_depths.npz $REPO/vendor/Lite-Mono/splits/eigen/gt_depths.npz
cd $REPO && python scripts/prepare_kitti.py --kitti_out $KITTI  # 파일 점검만 (모두 완료 표시되어 다운로드 없음)

## F. 공개 checkpoint 평가 (재현 점검)
공식 README 값: `0.107 0.765 4.561 0.183 0.886 0.963 0.983`. 두 번째 줄(이미지별 평가)은 첫 줄과 같아야 한다.

In [ ]:
!cd $REPO/vendor/Lite-Mono && python evaluate_depth.py --load_weights_folder $DRIVE/pretrained/lite-mono_640x192 --data_path $KITTI --model lite-mono
!cd $REPO && python analysis/eval_per_image.py eval --split_file data/splits/test_files.txt --gt $DRIVE/data/test_gt.npz --data_path $KITTI --weights $DRIVE/pretrained/lite-mono_640x192

## G. Baseline 학습 (끊기면 다시 실행 = 이어서 학습)
논문·공식 README 설정: ImageNet 초기화, batch 12, 30 epochs, lr `0.0001 5e-6 31 0.0001 1e-5 31`, 640x192, eigen_zhou.
`options.py` 기본값(batch 16, 50 epochs)과 다르므로 모두 명시한다. `NUM_WORKERS`는 증강 난수에 영향을 주므로 B/I/C에서 같게 둔다.

In [ ]:
RUN_ID = '20261115_baseline_s42'  # YYYYMMDD_method_seed, 처음 시작한 날짜로 고정
SEED, NUM_WORKERS = 42, 8
os.environ.update(RUN_ID=RUN_ID, SEED=str(SEED), NUM_WORKERS=str(NUM_WORKERS))

In [ ]:
%%bash
cd $REPO/vendor/Lite-Mono
RUN=$DRIVE/outputs/$RUN_ID; mkdir -p $RUN
LAST=$(ls $RUN/models/weights_*/train_state.json 2>/dev/null | sort -V | tail -1 | xargs -r dirname)
if [ -n "$LAST" ]; then INIT="--load_weights_folder $LAST"; else INIT="--mypretrain $DRIVE/pretrained/lite-mono-pretrain.pth"; fi
echo "$(date) start: $INIT" | tee -a $RUN/sessions.txt
nohup nvidia-smi --query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu --format=csv,noheader -l 60 >> $RUN/gpu_log.csv 2>&1 &
SMI=$!
python train.py --data_path $KITTI --log_dir $DRIVE/outputs --model_name $RUN_ID --model lite-mono --split eigen_zhou \
  --height 192 --width 640 --batch_size 12 --num_epochs 30 --lr 0.0001 5e-6 31 0.0001 1e-5 31 \
  --seed $SEED --num_workers $NUM_WORKERS $INIT 2>&1 | tee -a $RUN/train_log.txt
kill $SMI
echo "$(date) end" | tee -a $RUN/sessions.txt

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $DRIVE/outputs

## H. Checkpoint 선택 (val) → 최종 test
모든 epoch를 val 4,424장에서 이미지별로 평가한다(`per_image_val_files.csv`가 각 weights 폴더에 저장됨).
선택 규칙은 결과를 보기 전에 PLAN.md에 고정한다. 주의: 공식 val 프레임 대부분은 train 프레임의 반대 카메라 영상이다.

In [ ]:
%%bash
cd $REPO
python analysis/eval_per_image.py eval --split_file data/splits/val_files.txt --gt $DRIVE/data/val_gt.npz --data_path $KITTI \
  --weights $(ls -d $DRIVE/outputs/$RUN_ID/models/weights_* | sort -V) | tee $DRIVE/outputs/$RUN_ID/val_summary.txt

In [ ]:
CHOSEN_EPOCH = 'weights_29'  # 사전에 정한 규칙으로 고른 epoch
os.environ['CHOSEN'] = f"{DRIVE}/outputs/{RUN_ID}/models/{CHOSEN_EPOCH}"
!cd $REPO/vendor/Lite-Mono && python evaluate_depth.py --load_weights_folder $CHOSEN --data_path $KITTI --model lite-mono | tee $CHOSEN/eval_test_official.txt
!cd $REPO && python analysis/eval_per_image.py eval --split_file data/splits/test_files.txt --gt $DRIVE/data/test_gt.npz --data_path $KITTI --weights $CHOSEN